# Experiment 5: DiagScale transfer and retention

Evaluate the three DiagScale rates against their own original checkpoints using five outer folds on excluded credit tables and the fixed non-credit panel. All rates are reported. Original and adapter inference share precision settings. Earlier full-parameter CPT used other precision, episode and sampling settings and is not treated as a one-factor control here.

Can restricting cross-table credit adaptation to query scaling preserve useful in-context behavior? This is fixed-endpoint domain CPT, not target-table validation-selected fine-tuning.


## 0. Evidence and scope

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
from src.visualize import adapter_study as ds, publication as pub, comparisons as cmp, dashboards as db
sink = FigureSaver('experiment5/02_results')
report = cp.NotebookReport('Experiment 5: DiagScale transfer and retention', sink=sink)
print(source_description())
report.add('0. Evidence and scope', source_description() + '\n' + 'Can restricting cross-table credit adaptation to query scaling preserve useful in-context behavior? This is fixed-endpoint domain CPT, not target-table validation-selected fine-tuning.')
runs = [cp.load_campaign(5, track) for track in ('pd', 'lgd')]
from src.visualize import prediction_diagnostics as pred


DATA: output CreditPFN.zip; modified 2026-10-16T07:14:04+00:00; read only.
Project: output CreditPFN; modified 2026-10-16T08:49:22+00:00; read only.
Missing project records are unavailable evidence, not zero effects or failed training.


## 1. Checkpoint and benchmark coverage

In [2]:
for run in runs:
    report.table(run.track.upper()+' evidence', pub.availability(run))
    report.table(run.track.upper()+' paired benchmark coverage', cmp.benchmark_coverage(run))
report.add('1. Checkpoint and benchmark coverage', 'Trained and original cells need all five finite outer folds. Non-credit table comparisons additionally require all four training-dataset partitions. Missing cells remain unavailable.')

,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,24,DATA manifests
2,Epoch rows,18888,Project training diagnostics
3,Milestone rows,144,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,domain,base,planned_checkpoints,completed_checkpoints,paired_five_fold_comparisons,paired_datasets
0,credit,TabICLv2,12,12,0,0
1,credit,TabPFN v3,12,12,0,0
2,ood,TabICLv2,12,12,0,0
3,ood,TabPFN v3,12,12,0,0


,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,24,DATA manifests
2,Epoch rows,15012,Project training diagnostics
3,Milestone rows,54,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,domain,base,planned_checkpoints,completed_checkpoints,paired_five_fold_comparisons,paired_datasets
0,credit,TabICLv2,12,12,0,0
1,credit,TabPFN v3,12,12,0,0
2,ood,TabICLv2,12,12,0,0
3,ood,TabPFN v3,12,12,0,0


## 2. Five-fold credit transfer and non-credit retention

Points show tables; diamonds show their mean and intervals their interquartile range. All three rates share a table cohort within each model/domain panel. PD uses AUC and LGD uses R².

In [3]:
for run in runs:
    cp.show(sink, ds.plot_benchmark(run))
    for domain in ('credit', 'ood'):
        report.table(run.track.upper()+' '+domain+' paired primary scores', ds.benchmark_tables(run, domain=domain))
    report.table(run.track.upper()+' absolute scores including available classical controls', ds.absolute_scores(run))
report.add('2. Five-fold credit transfer and non-credit retention', 'Positive effects favor DiagScale. Table spread is not a confidence interval. The highest observed rate is not a validated selected recipe. Classical models provide benchmark context rather than a checkpoint-paired causal contrast.')

No measurements available for this section yet.


""


""


""


No measurements available for this section yet.


""


""


""


## 3. Probability quality and regression errors

Saved raw test predictions additionally measure systematic over/underprediction and paired prediction changes. These diagnostics describe observed errors; they do not identify theoretical bias or context variance.


In [4]:
for run in runs:
    if run.track == 'pd':
        cp.show(sink, ds.plot_benchmark(run, metric='brier_score'))
    for metric in (('brier_score', 'log_loss', 'ece', 'f1') if run.track == 'pd' else ('rmse', 'mae')):
        for domain in ('credit', 'ood'):
            report.table(run.track.upper()+' '+domain+' '+metric, ds.benchmark_tables(run, metric, domain))
for run in runs:
    prediction_values, prediction_coverage = pred.analyze(run)
    report.table(run.track.upper()+' saved prediction coverage', prediction_coverage)
    report.table(run.track.upper()+' paired per-table prediction diagnostics', prediction_values)
    cp.show(sink, pred.plot_diagnostics(prediction_values, run.track, coverage=prediction_coverage))
report.add('3. Probability quality and regression errors', 'Probabilities are evaluated at test prevalence. F1/calibration rules use inner validation only. RMSE effects are fractional reductions relative to the matched original; other differences retain their metric scale. Unsupported/nonfinite metrics are not filled with zeros.' + '\n' + pred.NOTE)


No measurements available for this section yet.


""


""


""


""


...

""


No measurements available for this section yet.


,status,reason
0,unavailable,No benchmark records


""


No measurements available for this section yet.


## 4. Cost and interpretation boundaries

In [5]:
for run in runs:
    report.table(run.track.upper()+' training resources', ds.costs(run))
report.add('4. Cost and interpretation boundaries', 'This experiment tests a cross-table DiagScale recipe. It is not a literal reproduction of target-table adapter fine-tuning, nor a pure parameter-count ablation against previous full-update runs. Overlapping training corpora, related data sources and the small retention panel limit independence and generality claims. Report unfavorable outcomes alongside improvements.')

,trial_name,base,learning_rate,partition,status,optimizer_selected_params,total_params,rows_seen,elapsed_sec,gpu_hours,peak_gpu_gb,query_scale_delta_rms
0,cpt_diagscale_v1_s00_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0001,0,OK,13056,53166200,70788725,6168.156499,1.713377,39.133,0.032038
1,cpt_diagscale_v1_s00_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0010,0,OK,13056,53166200,70788725,6147.738720,1.707705,39.345,0.184803
2,cpt_diagscale_v1_s00_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0300,0,OK,13056,53166200,70788725,6169.156107,1.713654,39.133,1.263868
3,cpt_diagscale_v1_s00_pd_tabicl-classifier-v2-2...,TabICLv2,0.0001,0,OK,7296,27559554,70788725,4766.576668,1.324049,15.873,0.038982
4,cpt_diagscale_v1_s00_pd_tabicl-classifier-v2-2...,TabICLv2,0.0010,0,OK,7296,27559554,70788725,4770.898052,1.325249,15.873,0.275439
5,cpt_diagscale_v1_s00_pd_tabicl-classifier-v2-2...,TabICLv2,0.0300,0,OK,7296,27559554,70788725,4776.914145,1.326921,15.763,1.640877
6,cpt_diagscale_v1_s01_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0001,1,OK,13056,53166200,69239144,6049.478816,1.680411,39.771,0.033577
7,cpt_diagscale_v1_s01_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0010,1,OK,13056,53166200,69239144,6071.310221,1.686475,39.558,0.199136
8,cpt_diagscale_v1_s01_pd_tabpfn-v3-classifier-v...,TabPFN v3,0.0300,1,OK,13056,53166200,69239144,6096.050658,1.693347,39.771,1.233613
9,cpt_diagscale_v1_s01_pd_tabicl-classifier-v2-2...,TabICLv2,0.0001,1,OK,7296,27559554,69239144,4809.640543,1.336011,15.873,0.053894


,trial_name,base,learning_rate,partition,status,optimizer_selected_params,total_params,rows_seen,elapsed_sec,gpu_hours,peak_gpu_gb,query_scale_delta_rms
0,cpt_diagscale_v1_s00_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0001,0,OK,12672,58287616,40087565,3079.852087,0.855514,23.140,0.080004
1,cpt_diagscale_v1_s00_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0010,0,OK,12672,58287616,40087565,3075.246109,0.854235,23.140,0.373031
2,cpt_diagscale_v1_s00_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0300,0,OK,12672,58287616,40087565,3096.738649,0.860205,23.140,2.073015
3,cpt_diagscale_v1_s00_lgd_tabicl-regressor-v2-2...,TabICLv2,0.0001,0,OK,7296,28552287,40087565,2277.811915,0.632726,6.741,0.056318
4,cpt_diagscale_v1_s00_lgd_tabicl-regressor-v2-2...,TabICLv2,0.0010,0,OK,7296,28552287,40087565,2279.549649,0.633208,6.741,0.354603
5,cpt_diagscale_v1_s00_lgd_tabicl-regressor-v2-2...,TabICLv2,0.0300,0,OK,7296,28552287,40087565,2282.008839,0.633891,6.627,2.086672
6,cpt_diagscale_v1_s01_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0001,1,OK,12672,58287616,24755261,1964.853834,0.545793,21.967,0.083976
7,cpt_diagscale_v1_s01_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0010,1,OK,12672,58287616,24755261,1969.742728,0.547151,21.967,0.398106
8,cpt_diagscale_v1_s01_lgd_tabpfn-v3-regressor-v...,TabPFN v3,0.0300,1,OK,12672,58287616,24755261,1951.440194,0.542067,21.967,2.063494
9,cpt_diagscale_v1_s01_lgd_tabicl-regressor-v2-2...,TabICLv2,0.0001,1,OK,7296,28552287,24755261,1577.989593,0.438330,6.838,0.053744


## Complete text summary

All displayed numerical values and captions, in section order.

In [6]:
print(report.summary(sink))

Experiment 5: DiagScale transfer and retention

0. Evidence and scope
DATA: output CreditPFN.zip; modified 2026-10-16T07:14:04+00:00; read only.
Project: output CreditPFN; modified 2026-10-16T08:49:22+00:00; read only.
Missing project records are unavailable evidence, not zero effects or failed training.
Can restricting cross-table credit adaptation to query scaling preserve useful in-context behavior? This is fixed-endpoint domain CPT, not target-table validation-selected fine-tuning.

1. Checkpoint and benchmark coverage
Trained and original cells need all five finite outer folds. Non-credit table comparisons additionally require all four training-dataset partitions. Missing cells remain unavailable.

Table: PD evidence
CSV (first column: row index)
row,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,24,DATA manifests
2,Epoch rows,18888,Project training diagnostics
3,Milestone rows,144,Project trajectory diagnostics
4,Benchmark folds,0,Proje